# 🔧 Building a Basic RAG System

**Day 7 — Notebook 2 of 4 | Estimated Time: 40 minutes**

---

## What You'll Learn
- Build a complete RAG system from scratch using Gemini + ChromaDB
- Implement the full pipeline: index → query → retrieve → generate
- Add source citations to generated answers
- Handle multi-turn conversations with RAG context
- Measure basic retrieval quality

---

## Setup

In [ ]:
%pip install -U -q "google-genai>=1.0.0" chromadb langchain-text-splitters

In [ ]:
import sys, os, hashlib
import numpy as np
import chromadb
from langchain_text_splitters import RecursiveCharacterTextSplitter

repo_root = os.path.abspath(os.path.join(os.getcwd(), "../../.."))
if repo_root not in sys.path:
    sys.path.append(repo_root)

from helpers.auth import get_api_key
from google import genai
from google.genai import types
from IPython.display import Markdown

client = genai.Client(api_key=get_api_key())
EMBEDDING_MODEL = "text-embedding-004"
GEN_MODEL = "gemini-2.5-flash"
print("✅ Ready!")

---

## 1. The Knowledge Base

We'll build a RAG system over a fictional company's employee handbook — a realistic use case.

In [ ]:
EMPLOYEE_HANDBOOK = [
    {
        "title": "Welcome and Company Mission",
        "content": """Welcome to Acme Corp! Founded in 2010, Acme Corp is a technology company focused on 
building tools that help teams collaborate more effectively. Our mission is to make work more 
human. We believe that great software should get out of the way and let people do their best work.
Our core values are: transparency, ownership, and continuous improvement. We operate across 
offices in London, New York, and Singapore, with over 800 employees worldwide.""",
    },
    {
        "title": "Working Hours and Flexibility",
        "content": """Acme Corp operates on a flexible working model. Core hours are 10am–3pm in your 
local timezone, during which all team members are expected to be available for meetings and 
collaboration. Outside core hours, employees manage their own schedules. The standard working 
week is 40 hours. Overtime is discouraged — if you're regularly working more than 40 hours, 
speak to your manager about workload management. Part-time arrangements (minimum 3 days/week) 
can be arranged with manager and HR approval.""",
    },
    {
        "title": "Annual Leave Policy",
        "content": """All full-time employees receive 25 days of annual leave per year, in addition to 
public holidays. Leave accrues from the first day of employment. Up to 5 days of unused leave 
may be carried over to the following year; any excess is forfeited. Leave must be approved by 
your line manager at least 10 working days in advance for periods of 3+ days. Single days can 
be requested with 2 days' notice. Sick leave is separate and unlimited, subject to a fit-to-work 
note after 7 consecutive days.""",
    },
    {
        "title": "Remote Work Policy",
        "content": """Employees are expected to work from the office a minimum of 2 days per week. 
The remaining days may be worked remotely. Extended remote work (more than 2 consecutive weeks 
from a location other than your contracted office) requires manager and HR approval. The company 
provides a home office allowance of £600 per year for equipment and ergonomic furniture. 
Employees working remotely must have a stable internet connection of at least 50 Mbps and a 
private space suitable for confidential calls.""",
    },
    {
        "title": "Performance Reviews",
        "content": """Acme Corp runs a biannual performance review cycle — in June and December. 
Reviews involve a self-assessment, manager assessment, and 360-degree peer feedback. 
Performance is rated on a 5-point scale: Exceptional, Exceeds Expectations, Meets Expectations, 
Developing, and Unsatisfactory. Salary increases are tied to the December review cycle. 
Promotions can happen at any time but are most commonly aligned with the December review. 
New employees complete a 90-day onboarding review before entering the standard cycle.""",
    },
    {
        "title": "Learning and Development",
        "content": """Each employee has an annual learning budget of £1,500 for courses, books, 
conferences, and certifications. Requests must be approved by your manager before purchase. 
Acme Corp also provides free access to O'Reilly Learning and LinkedIn Learning. 
The company hosts a monthly internal Tech Talk series where engineers share knowledge. 
Up to 2 days per quarter may be used for learning activities during work hours with manager 
approval. All learning activities should be logged in the HR system within 30 days of completion.""",
    },
    {
        "title": "Code of Conduct",
        "content": """Acme Corp is committed to maintaining a respectful and inclusive workplace. 
Harassment, discrimination, and bullying of any kind will not be tolerated. This policy applies 
to all employees, contractors, and visitors, both in the office and in remote or virtual settings. 
Any incidents should be reported to your HR Business Partner or via the anonymous reporting 
hotline at ethics.acmecorp.com. All reports are treated confidentially. Retaliation against 
anyone who reports a concern in good faith is a disciplinary offence.""",
    },
    {
        "title": "Equipment and IT Policy",
        "content": """All employees receive a company laptop and are eligible to request a monitor, 
keyboard, and mouse. Equipment is refreshed on a 3-year cycle. Personal use of company equipment 
is permitted within reason, but must not compromise security or productivity. Company data must 
never be stored on personal devices or unencrypted external drives. All devices must have 
full-disk encryption enabled. Software installations must be approved by IT — use the self-service 
portal at it.acmecorp.com for requests. Equipment issues should be reported to helpdesk@acmecorp.com.""",
    },
    {
        "title": "Benefits and Compensation",
        "content": """Acme Corp offers a comprehensive benefits package. Health insurance is provided 
for all employees and their dependents at no cost. The company contributes 6% of salary to a 
pension/401k plan, with employee contributions matched up to 6%. Employees receive a £50/month 
wellness allowance for gym memberships, fitness apps, or mental health services. 
Share options are granted to all permanent employees after 6 months, vesting over 4 years 
with a 1-year cliff. Salary reviews happen annually in December alongside performance reviews.""",
    },
    {
        "title": "Parental Leave",
        "content": """Acme Corp provides enhanced parental leave. Primary caregivers receive 26 weeks 
of fully paid parental leave, followed by up to 26 weeks of statutory pay. Secondary caregivers 
receive 8 weeks of fully paid leave. Adoption leave follows the same structure as parental leave. 
Employees returning from parental leave are guaranteed their original role or an equivalent 
position at the same level and pay. A keep-in-touch (KIT) allowance of up to 10 days per year 
is available for employees on parental leave who wish to attend key meetings or training.""",
    },
]

print(f"Knowledge base: {len(EMPLOYEE_HANDBOOK)} sections")
total_words = sum(len(s['content'].split()) for s in EMPLOYEE_HANDBOOK)
print(f"Total words: {total_words}")

---

## 2. Build the RAG Class

In [ ]:
class SimpleRAG:
    """
    A minimal but complete RAG system.
    Handles indexing, retrieval, and generation in one class.
    """

    SYSTEM_PROMPT = """You are a helpful HR assistant for Acme Corp. Answer employee questions 
accurately using only the provided handbook sections. Be concise and friendly.
If the answer is not covered in the provided context, say: 
"I don't have that information in the handbook. Please contact hr@acmecorp.com."
Always cite which section your answer comes from."""

    RAG_TEMPLATE = """\
The following sections from the Acme Corp Employee Handbook are relevant to the question:

{context}

Employee question: {question}"""

    def __init__(self, chunk_size: int = 400, chunk_overlap: int = 60, top_k: int = 3):
        self.top_k = top_k
        self.chroma = chromadb.Client()
        self.collection = self.chroma.create_collection(
            name="handbook",
            metadata={"hnsw:space": "cosine"},
        )
        self.splitter = RecursiveCharacterTextSplitter(
            chunk_size=chunk_size,
            chunk_overlap=chunk_overlap,
        )

    # ── Indexing ──────────────────────────────────────────────────────────────

    def index(self, documents: list[dict]) -> None:
        """Load, chunk, embed, and store all documents."""
        all_chunks = []
        for doc in documents:
            for i, chunk_text in enumerate(self.splitter.split_text(doc["content"])):
                chunk_id = hashlib.md5(f"{doc['title']}_{i}".encode()).hexdigest()[:10]
                all_chunks.append({
                    "id": chunk_id,
                    "text": chunk_text,
                    "title": doc["title"],
                    "chunk_index": i,
                })

        # Batch embed
        texts = [c["text"] for c in all_chunks]
        result = client.models.embed_content(
            model=EMBEDDING_MODEL,
            contents=texts,
            config=types.EmbedContentConfig(task_type="RETRIEVAL_DOCUMENT"),
        )
        embeddings = [e.values for e in result.embeddings]

        # Store
        self.collection.add(
            ids=[c["id"] for c in all_chunks],
            documents=texts,
            embeddings=embeddings,
            metadatas=[{"title": c["title"], "chunk_index": c["chunk_index"]} for c in all_chunks],
        )
        print(f"✅ Indexed {len(all_chunks)} chunks from {len(documents)} documents.")

    # ── Retrieval ─────────────────────────────────────────────────────────────

    def retrieve(self, query: str) -> list[dict]:
        """Embed the query and retrieve the top-K most relevant chunks."""
        q_result = client.models.embed_content(
            model=EMBEDDING_MODEL,
            contents=query,
            config=types.EmbedContentConfig(task_type="RETRIEVAL_QUERY"),
        )
        q_vec = q_result.embeddings[0].values

        results = self.collection.query(
            query_embeddings=[q_vec],
            n_results=self.top_k,
            include=["documents", "metadatas", "distances"],
        )

        return [
            {
                "text": results["documents"][0][i],
                "title": results["metadatas"][0][i]["title"],
                "score": 1 - results["distances"][0][i],
            }
            for i in range(len(results["ids"][0]))
        ]

    # ── Generation ────────────────────────────────────────────────────────────

    def generate(self, query: str, chunks: list[dict]) -> str:
        """Build the augmented prompt and generate a grounded answer."""
        context_parts = []
        for chunk in chunks:
            context_parts.append(f"[Section: {chunk['title']}]\n{chunk['text']}")
        context = "\n\n---\n\n".join(context_parts)

        prompt = self.RAG_TEMPLATE.format(context=context, question=query)

        response = client.models.generate_content(
            model=GEN_MODEL,
            contents=prompt,
            config=types.GenerateContentConfig(
                system_instruction=self.SYSTEM_PROMPT,
                temperature=0.1,
            ),
        )
        return response.text

    # ── Full pipeline ─────────────────────────────────────────────────────────

    def ask(self, question: str, verbose: bool = False) -> str:
        """Retrieve + Generate: the main RAG entry point."""
        chunks = self.retrieve(question)

        if verbose:
            print(f"📚 Retrieved {len(chunks)} chunks:")
            for c in chunks:
                print(f"   [{c['score']:.3f}] {c['title']}: {c['text'][:60]}...")
            print()

        return self.generate(question, chunks)


# Initialise and index
rag = SimpleRAG(chunk_size=400, chunk_overlap=60, top_k=3)
rag.index(EMPLOYEE_HANDBOOK)

---

## 3. Ask Questions

In [ ]:
# Basic Q&A
questions = [
    "How many days of annual leave do I get?",
    "Can I work from home every day?",
    "What happens to my unused leave at the end of the year?",
    "How much money do I have for learning and training?",
    "When are performance reviews held?",
]

for q in questions:
    print(f"❓ {q}")
    answer = rag.ask(q)
    print(f"🤖 {answer.strip()}")
    print("-" * 60)

In [ ]:
# Ask with verbose retrieval details
print("=== VERBOSE MODE: See what was retrieved ===\n")
answer = rag.ask("What is the parental leave for a secondary caregiver?", verbose=True)
print(f"🤖 Answer:\n{answer.strip()}")

In [ ]:
# Out-of-scope question — should refuse gracefully
print("=== OUT-OF-SCOPE QUESTION ===\n")
answer = rag.ask("What is the stock price of Acme Corp today?")
print(f"🤖 {answer.strip()}")

---

## 4. Multi-Turn Conversation with RAG

In a real chatbot, questions build on previous context. We need to maintain conversation history and re-query for each turn.

In [ ]:
class RAGChatbot:
    """Multi-turn RAG chatbot with conversation history."""

    def __init__(self, rag: SimpleRAG):
        self.rag = rag
        self.history = []  # List of {role, content} dicts

    def chat(self, user_message: str) -> str:
        # Retrieve relevant context for this turn
        chunks = self.rag.retrieve(user_message)
        context = "\n\n".join(
            f"[{c['title']}] {c['text']}" for c in chunks
        )

        # Build conversation history as a formatted string
        history_text = ""
        if self.history:
            history_text = "\nPrevious conversation:\n"
            for msg in self.history[-4:]:  # Last 2 turns
                history_text += f"{msg['role'].capitalize()}: {msg['content']}\n"

        prompt = f"""Handbook context:\n{context}
{history_text}
Employee: {user_message}"""

        response = client.models.generate_content(
            model=GEN_MODEL,
            contents=prompt,
            config=types.GenerateContentConfig(
                system_instruction=self.rag.SYSTEM_PROMPT,
                temperature=0.1,
            ),
        )
        answer = response.text.strip()

        # Store in history
        self.history.append({"role": "user", "content": user_message})
        self.history.append({"role": "assistant", "content": answer})

        return answer

    def reset(self):
        self.history = []


bot = RAGChatbot(rag)

# Simulate a multi-turn conversation
conversation = [
    "How much annual leave do I get?",
    "Can I carry any over to next year?",          # References previous answer
    "What about sick leave — is that separate?",   # Follow-up
]

print("=== MULTI-TURN CONVERSATION ===")
for turn in conversation:
    print(f"\n👤 Employee: {turn}")
    response = bot.chat(turn)
    print(f"🤖 HR Bot: {response}")

---

## 5. Evaluating Retrieval Quality

In [ ]:
# Golden evaluation set: (query, expected_section_title)
eval_set = [
    ("How many days off do I get per year?", "Annual Leave Policy"),
    ("Can I work from home?", "Remote Work Policy"),
    ("When are salary increases decided?", "Performance Reviews"),
    ("What is the learning budget?", "Learning and Development"),
    ("What do I do if I'm harassed at work?", "Code of Conduct"),
    ("How does the pension scheme work?", "Benefits and Compensation"),
    ("What are core working hours?", "Working Hours and Flexibility"),
    ("How long is maternity leave?", "Parental Leave"),
]

hits_at_1 = 0
hits_at_3 = 0

print("Retrieval Evaluation:\n")
for query, expected_title in eval_set:
    chunks = rag.retrieve(query)
    retrieved_titles = [c["title"] for c in chunks]

    hit1 = retrieved_titles[0] == expected_title
    hit3 = expected_title in retrieved_titles
    hits_at_1 += hit1
hits_at_3 += hit3

    icon = "✅" if hit1 else ("⚠️" if hit3 else "❌")
    print(f"  {icon} Expected: {expected_title}")
    print(f"     Got:      {retrieved_titles[0]} (top-1)")

print(f"\n📊 Precision@1: {hits_at_1}/{len(eval_set)} = {hits_at_1/len(eval_set):.0%}")
print(f"📊 Recall@3:    {hits_at_3}/{len(eval_set)} = {hits_at_3/len(eval_set):.0%}")

---

## 🏋️ Exercise 1: Extend the Knowledge Base

Add new documents and test that they are retrievable.

In [ ]:
# Exercise 1: Add new documents
# TODO:
# 1. Write 2 new handbook sections on topics not yet covered
#    (e.g., Travel Policy, Office Facilities, Onboarding Process)
# 2. Add them to the RAG system using rag.collection.upsert()
#    (chunk them first using the splitter)
# 3. Ask questions that can only be answered from your new sections
# 4. Verify the answers are correct

new_sections = [
    {
        "title": "TODO: Your section title",
        "content": "TODO: Your section content (150-300 words)",
    },
]

# TODO: Chunk, embed, and upsert the new sections
# Then test with: rag.ask("Your question about the new section")

---

## 🏋️ Exercise 2: Tune top_k

Does retrieving more chunks improve answer quality? Find the sweet spot.

In [ ]:
# Exercise 2: Tune top_k
# TODO:
# This question requires information from TWO sections (Annual Leave + Working Hours)
# Try top_k = 1, 2, 3, 5 and observe whether the answer becomes more complete

cross_section_question = "I want to take a long break — how many days do I get each year, "\
                         "and do I have to give notice? Also, can I work part-time instead?"

for k in [1, 2, 3, 5]:
    rag_k = SimpleRAG(top_k=k)
    rag_k.index(EMPLOYEE_HANDBOOK)
    
    print(f"\n=== top_k={k} ===")
    chunks = rag_k.retrieve(cross_section_question)
    print(f"Retrieved sections: {[c['title'] for c in chunks]}")
    answer = rag_k.generate(cross_section_question, chunks)
    print(f"Answer: {answer.strip()[:300]}...")

---

## Key Takeaways

| Component | What it does |
|-----------|-------------|
| `index()` | Chunk → embed → store in ChromaDB |
| `retrieve()` | Embed query → ANN search → return top-K chunks |
| `generate()` | Inject chunks into prompt → call Gemini → return grounded answer |
| `top_k` | More chunks = more context, but also more noise — tune per use case |
| System prompt | Critical for enforcing grounding — "use only the context" |
| Evaluation | Precision@K measures how often the right chunk is in the top results |

---

## 📚 Further Reading

| Resource | Type | Link |
|----------|------|------|
| RAG Paper | Paper | [arxiv.org/abs/2005.11401](https://arxiv.org/abs/2005.11401) |
| ChromaDB Docs | Docs | [docs.trychroma.com](https://docs.trychroma.com/) |

---

**Next up:** [03_RAG_with_LangChain.ipynb](./03_RAG_with_LangChain.ipynb)